In [30]:
# Parameters
DB_PATH          = "../../../DB/oedb_baseline_v3.db"
BENCHMARK_PATH   = "../../../data/input_data/benchmark_trainingset.xlsx"
BENCHMARK_SHEET  = "questions"
MATCHED_CSV_PATH = "matched_questions.csv"
NOTEGROUP_ID_MIN = 1
NOTEGROUP_ID_MAX = 23
MATCH_THRESHOLD  = 85

WEIGHTS = {
    "question_content":          95,
    "main_indicator":             1,
    "followed_question_content":  1,
}


In [31]:
import sqlite3
import pandas as pd
from rapidfuzz import fuzz
import re

pd.reset_option("display.max_rows")
pd.reset_option("display.max_colwidth")

def load_etl(db_path, id_min, id_max):
    con = sqlite3.connect(db_path)
    df = pd.read_sql_query(
        """SELECT q.questionID, q.notegroupID, q.question_content,
                  q.main_indicator, q.followed_questionID,
                  p.question_content AS followed_question_content
           FROM questions q
           LEFT JOIN questions p ON q.followed_questionID = p.questionID
           WHERE q.notegroupID BETWEEN ? AND ?""",
        con, params=(id_min, id_max)
    )
    con.close()
    return df

def load_benchmark(xlsx_path, sheet, id_min, id_max):
    df = pd.read_excel(xlsx_path, sheet_name=sheet, dtype=str)
    df["notegroupID"] = df["notegroupID"].astype(int)
    df = df[df["notegroupID"].between(id_min, id_max)]
    # self-join to resolve followed_question_content
    fq = df[["questionID", "question_content"]].rename(columns={
        "questionID":       "followed_questionID",
        "question_content": "followed_question_content"
    })
    df = df.merge(fq, on="followed_questionID", how="left")
    return df

etl = load_etl(DB_PATH, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)
bm  = load_benchmark(BENCHMARK_PATH, BENCHMARK_SHEET, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)

print("ETL records:      ", len(etl))
print("Benchmark records:", len(bm))

ETL records:       420
Benchmark records: 427


In [32]:
def normalise_str(val):
    if pd.isna(val) or str(val).strip() in ("", "None", "nan"):
        return None
    s = str(val).strip().lower()
    s = re.sub(r'\s*\n\s*', '\n', s)  # normalise whitespace around newlines
    return s

def question_pair_score(etl_row, bm_row):
    total_weight = 0
    weighted_sum = 0.0
    for field, weight in WEIGHTS.items():
        e = normalise_str(etl_row.get(field))
        b = normalise_str(bm_row.get(field))
        if e is None or b is None:
            continue   # exclude null fields from weighted average
        sim = fuzz.ratio(e, b)
        weighted_sum += sim * weight
        total_weight += weight
    if total_weight == 0:
        return 0.0
    return weighted_sum / total_weight

In [33]:

def is_substring_match(etl_row, bm_row, threshold):
    """
    Secondary match condition: one question_content is a substring of the other,
    AND supporting fields (main_indicator, followed_question_content) reach threshold
    when available. If both supporting fields are null on both sides, also matched.
    """
    e = normalise_str(etl_row.get("question_content"))
    b = normalise_str(bm_row.get("question_content"))
    if e is None or b is None:
        return False

    # Check substring in either direction
    if e not in b and b not in e:
        return False

    # Evaluate supporting fields
    support_fields = ["main_indicator", "followed_question_content"]
    total_weight = 0
    weighted_sum = 0.0

    for field in support_fields:
        ev = normalise_str(etl_row.get(field))
        bv = normalise_str(bm_row.get(field))
        if ev is None or bv is None:
            continue
        weighted_sum += fuzz.ratio(ev, bv) * 0.5
        total_weight += 0.5

    if total_weight == 0:
        # Both supporting fields null on both sides → matched
        return True

    return (weighted_sum / total_weight) >= threshold

def map_questions(etl_df, bm_df, threshold):
    """
    Match questions within each notegroupID by weighted similarity.
    Fields: question_content (98%), main_indicator (1%), followed_question_content (1%).
    Null fields are excluded from the weighted average.
    Returns:
        matched  : list of (etl_idx, bm_idx, score)
        etl_only : list of etl_idx  → FP rows
        bm_only  : list of bm_idx   → FN rows
    """
    matched  = []
    etl_only = []
    bm_only  = []

    for ng_id in sorted(etl_df["notegroupID"].unique()):
        etl_ng = etl_df[etl_df["notegroupID"] == ng_id]
        bm_ng  = bm_df[bm_df["notegroupID"]  == ng_id]

        if bm_ng.empty:
            etl_only.extend(etl_ng.index.tolist())
            continue
        if etl_ng.empty:
            bm_only.extend(bm_ng.index.tolist())
            continue

        scores = {}
        for ei in etl_ng.index:
            for bi in bm_ng.index:
                primary_score = question_pair_score(etl_ng.loc[ei], bm_ng.loc[bi])
                # Secondary condition: substring match
                if primary_score < threshold and is_substring_match(etl_ng.loc[ei], bm_ng.loc[bi], threshold):
                    primary_score = threshold  # treat as exactly at threshold
                scores[(ei, bi)] = primary_score

        used_etl = set()
        used_bm  = set()
        for (ei, bi), score in sorted(scores.items(), key=lambda x: -x[1]):
            if score < threshold:
                break
            if ei in used_etl or bi in used_bm:
                continue
            matched.append((ei, bi, round(score, 2)))
            used_etl.add(ei)
            used_bm.add(bi)

        etl_only.extend([i for i in etl_ng.index if i not in used_etl])
        bm_only.extend( [i for i in bm_ng.index  if i not in used_bm])

    return matched, etl_only, bm_only


matched, etl_only, bm_only = map_questions(etl, bm, MATCH_THRESHOLD)

print(f"Matched pairs : {len(matched)}")
print(f"ETL-only (FP) : {len(etl_only)}")
print(f"BM-only  (FN) : {len(bm_only)}")

Matched pairs : 418
ETL-only (FP) : 2
BM-only  (FN) : 9


In [34]:
match_rows = []
for ei, bi, score in matched:
    match_rows.append({
        "notegroupID":    etl.loc[ei, "notegroupID"],
        "etl_questionID": etl.loc[ei, "questionID"],
        "bm_questionID":  bm.loc[bi, "questionID"] if "questionID" in bm.columns else None,
        "etl_question":   etl.loc[ei, "question_content"],
        "bm_question":    bm.loc[bi, "question_content"] if "question_content" in bm.columns else None,
        "score":          score,
    })

#pd.DataFrame(match_rows)

,notegroupID,etl_questionID,bm_questionID,etl_question,bm_question,score
0,1,1,1,How is your inburgering going so far?,How is your inburgering going so far?,100.00
1,1,3,3,"Do you work? If yes: is your work paid, or vol...","Do you work? If yes: is your work paid, or vol...",100.00
2,1,4,4,How do you feel about your work/volunteer work...,How do you feel about your work/volunteer work...,100.00
3,1,6,6,If you are working/doing volunteer work: Do yo...,If you are working/doing volunteer work: Do yo...,100.00
4,1,7,7,For everyone in the group: What is missing for...,For everyone in the group: What is missing for...,100.00
...,...,...,...,...,...,...
413,23,438,438,Terugkijken op het gesprek\nAls je terugkijkt ...,Terugkijken op het gesprek\n Als je terugkijkt...,100.00
414,23,439,439,Verbeterideeën\nStel dat jij één ding kon vera...,Verbeterideeën\n Stel dat jij één ding kon ver...,100.00
415,23,440,440,Toekomst en participatie\nWat heb je nodig om ...,Toekomst en participatie\n Wat heb je nodig om...,100.00
416,23,441,441,Blik van nieuwkomers\nWat zou jij willen dat N...,Blik van nieuwkomers\n Wat zou jij willen dat ...,100.00


In [35]:
print("=== ETL-only (FP) ===")
display(etl.loc[etl_only, ["notegroupID", "questionID", "question_content"]])

print("\n=== BM-only (FN) ===")
bm_cols = [c for c in ["notegroupID", "questionID", "question_content"] if c in bm.columns]
display(bm.loc[bm_only, bm_cols])

=== ETL-only (FP) ===


,notegroupID,questionID,question_content
378,22,399,Door ons werk leren we vooral veel dagelijkse ...
379,22,400,Voor ons horen werk en taallessen echt bij elk...



=== BM-only (FN) ===


,notegroupID,questionID,question_content
290,15,442,What did you learn about healthcare in The Net...
344,21,443,Wat betekent “goede inburgering” voor jou pers...
373,22,444,Past de manier van lesgeven bij jouw niveau en...
376,22,445,Hoe heb je het opgelost?
378,22,446,Te veel of te weinig informatie?
379,22,447,Welke contactvorm werkt het best?
380,22,448,Geeft dit rust of stress?
385,22,399,Leren door werk
386,22,400,Toekomst & aansluiting


In [36]:
csv_rows = []
for ei, bi, score in matched:
    csv_rows.append({
        "notegroupID":        etl.loc[ei, "notegroupID"],
        "etl_questionID":     etl.loc[ei, "questionID"],
        "bm_questionID":      bm.loc[bi, "questionID"] if "questionID" in bm.columns else None,
        "etl_question_content": etl.loc[ei, "question_content"],
        "bm_question_content":  bm.loc[bi, "question_content"] if "question_content" in bm.columns else None,
        "match_score":        score,
    })

matched_csv = pd.DataFrame(csv_rows)
matched_csv.to_csv(MATCHED_CSV_PATH, index=False)
print(f"Saved {len(matched_csv)} matched pairs to {MATCHED_CSV_PATH}")

Saved 418 matched pairs to matched_questions.csv
